# Exercise 3: Extracting tables from HTML pages

🤔 **Are people more depressed in countries with less sunshine?**

To answer this question, we need two pieces of data that are available on Wikipedia, but in two different pages, as **HTML tables**:
- [Epidemiology of depression](https://en.wikipedia.org/wiki/Epidemiology_of_depression): the depression rate of each country,
- [List of cities by sunshine duration](https://en.wikipedia.org/wiki/List_of_cities_by_sunshine_duration): the number of sunshine hours of many cities.

Both pages have been saved in the folder `data/1_data_collection/exercice_1/`, so that the exercise always works, even if Wikipedia changes.

| Step | What we do | Tool |
|---|---|---|
| 1 | Load and parse the two HTML pages | `BeautifulSoup` |
| 2 | Understand how an HTML table is built | `BeautifulSoup` |
| 3 | Extract the depression rate of each country | `BeautifulSoup` + `pandas` |
| 4 | Extract the average sunshine of each country | `BeautifulSoup` + `pandas` |
| 5 | Compare depression and sunshine | `pandas` + `seaborn` |

*This exercise is inspired by https://github.com/khuyentran1401/Web-Scrapping-Wikipedia.*

### How to work

- Run the cells **one by one, from top to bottom** (`Shift + Enter`).
- ✏️ marks a cell you have to complete: replace the lines marked `## @COMPLETE`.
- 🧪 marks a test cell: run it to check your answer before moving on.
- 💡 gives you a hint, 🤔 asks you a question to think about.

In [ ]:
from pathlib import Path

import pandas as pd
import seaborn as sns
from bs4 import BeautifulSoup

DATA_PATH = Path("../data/1_data_collection/exercice_1")

## Step 1: Load and parse the HTML pages

As in exercise 1, we read each HTML file as text and parse it with BeautifulSoup.

In [ ]:
def load_soup(filename):
    """Read an HTML file stored locally and parse it with BeautifulSoup."""
    with open(DATA_PATH / filename, encoding="utf-8") as file:
        return BeautifulSoup(file.read(), "html.parser")


depression_soup = load_soup("epidemiology_of_depression.html")
sunshine_soup = load_soup("city_sunshine_duration.html")

print(depression_soup.title.text)
print(sunshine_soup.title.text)

## Step 2: How is an HTML table built?

An HTML table is a `<table>` tag, made of **rows** (`<tr>`, *table row*), each row being made of **cells**: `<th>` (*table header*) for the column names, `<td>` (*table data*) for the values.

```html
<table>
  <tr> <th>Country</th>     <th>1990</th>   <th>2017</th>   ... </tr>
  <tr> <td>Afghanistan</td> <td>439.81</td> <td>443.57</td> ... </tr>
  <tr> <td>Albania</td>     <td>306.48</td> <td>308.44</td> ... </tr>
  ...
</table>
```

So, to read a table, we:
1. get its rows with `table.find_all("tr")`,
2. for each row, get its cells with `row.find_all(["td", "th"])` (both kinds of cells),
3. take the `.text` of the cells we are interested in.

The depression page contains a single data table: the first `<table>` of the page. Run the cell below to look at its first three rows.

In [ ]:
depression_table = depression_soup.find("table")

for row in depression_table.find_all("tr")[:3]:
    cells = row.find_all(["td", "th"])
    print([cell.text.strip() for cell in cells])

The first row contains the column names. The depression rate (measured in DALY, *disability-adjusted life years*, per 100,000 inhabitants) is given for 1990 and 2017. We keep:
- the **country**, in column `0`,
- the **rate of 2017**, in column `2`.

## Step 3: Extract the depression rates

✏️ **Complete the function `extract_depression_rates`**: get the rows of the table, then the cells of each row.

💡 *Hint*: look again at the cell of step 2, everything you need is there.

In [ ]:
COUNTRY_COLUMN = 0
RATE_COLUMN = 2


def extract_depression_rates(soup):
    """Return a DataFrame with the depression rate (DALY, 2017) of each country."""
    table = soup.find("table")
    rates = {}  # country -> depression rate

    ## @COMPLETE : get all the rows of the table
    rows = []
    for row in rows[1:]:  # skip the first row (column names)
        ## @COMPLETE : get all the cells of the row
        cells = []
        country = cells[COUNTRY_COLUMN].text.strip()
        rate = float(cells[RATE_COLUMN].text.strip())
        rates[country] = round(rate)

    return pd.DataFrame.from_dict(rates, orient="index", columns=["DALY rate"])

🧪 **Test**: you should get the depression rate of about 230 countries, starting with Afghanistan and Albania.

In [ ]:
df_depression = extract_depression_rates(depression_soup)

print("Depression rates extracted for", len(df_depression), "countries")
df_depression.head()

## Step 4: Extract the sunshine hours

The sunshine page is a bit more complex:
- it contains **several tables** (one per continent), and also some tables that are not data (warning boxes, menus, ...). The data tables are the ones with the class `wikitable`, so we select them with `soup.find_all("table", class_="wikitable")`,
- each row is a **city**, not a country:

| Country | City | Jan | Feb | ... | Dec | Year | Ref. |
|---|---|---|---|---|---|---|---|
| Ivory Coast | Gagnoa | 183.0 | 180.0 | ... | 164.0 | 1,823.0 | [2] |

We keep the country (column `0`) and the yearly sunshine hours (column `-2`, the second to last). Then, for each country, we compute the **average over its cities**, and divide by 12 to get a number of hours **per month**.

The yearly hours are written with a comma to separate thousands (`1,823.0`): the function `parse_number` removes it before converting the text into a number.

✏️ **Complete the function `extract_sunshine_hours`**: same two lines as in step 3!

In [ ]:
COUNTRY_COLUMN = 0
YEAR_COLUMN = -2


def parse_number(text):
    """Convert a number written with a comma between thousands into a float: "1,823.0" -> 1823.0"""
    return float(text.replace(",", ""))


def extract_sunshine_hours(soup):
    """Return a DataFrame with the average monthly sunshine hours of each country."""
    sunshine = {}  # country -> list of yearly sunshine hours (one value per city)

    for table in soup.find_all("table", class_="wikitable"):
        ## @COMPLETE : get all the rows of the table
        rows = []
        for row in rows[1:]:  # skip the first row (column names)
            ## @COMPLETE : get all the cells of the row
            cells = []
            country = cells[COUNTRY_COLUMN].text.strip()
            yearly_hours = parse_number(cells[YEAR_COLUMN].text.strip())

            if country not in sunshine:
                sunshine[country] = []
            sunshine[country].append(yearly_hours)

    # Average over the cities of each country, then convert into hours per month
    monthly = {country: round(sum(hours) / len(hours) / 12) for country, hours in sunshine.items()}

    return pd.DataFrame.from_dict(monthly, orient="index", columns=["Sunshine hours/month"])

🧪 **Test**: you should get the average sunshine of more than 100 countries.

In [ ]:
df_sunshine = extract_sunshine_hours(sunshine_soup)

print("Sunshine hours extracted for", len(df_sunshine), "countries")
df_sunshine.head()

## Step 5: Compare depression and sunshine

We now have two tables indexed by country. We **join** them, and keep only the countries present in both (`dropna` removes the rows with a missing value).

In [ ]:
df = df_depression.join(df_sunshine).dropna()

print("Both values are known for", len(df), "countries")
df.head()

Finally, we plot one point per country and compute the **Pearson correlation** between the two columns: it is close to `-1` if more sunshine goes with less depression, close to `+1` if more sunshine goes with more depression, and close to `0` if there is no linear relation.

In [ ]:
correlation = df["DALY rate"].corr(df["Sunshine hours/month"])

sns.scatterplot(data=df, x="Sunshine hours/month", y="DALY rate").set_title(
    f"Pearson correlation: {correlation:.2f}"
);

🤔 **Questions**:
- What does the correlation tell you? Are people more depressed in countries with less sunshine?
- Even if the correlation were strong, could we conclude that sunshine *causes* less depression? Which other factors could explain it?
- Some countries were lost in the join (compare the number of countries before and after). Why? (💡 *Hint*: compare how the country names are written in both tables.)

## 🎉 Well done!

You have turned two web pages into a dataset ready for analysis:
1. an HTML table is made of rows (`<tr>`) and cells (`<td>`, `<th>`),
2. `find_all("tr")` then `find_all(["td", "th"])` lets you read any table, cell by cell,
3. `pandas` then makes it easy to clean, join and analyse the data.